# Corte 1 — Prototipo Base
## Materia: Deep Learning · QA Test Case Generator con LLM Local

**Programa:** Ingeniería Mecatrónica / Electrónica  
**Modalidad:** Equipos de 2 personas  
**Repositorio:** https://github.com/JesusAndres2512/Proyecto_DL_QA  
**Fecha:** 2026-04-25

---

## 1. Definición del Caso de Uso

### Problema a resolver
Los equipos de QA invierten entre 2 y 6 horas redactando manualmente casos de prueba para cada historia de usuario. Este tiempo podría reducirse drásticamente usando un modelo de lenguaje local que genere automáticamente suites completas de test cases, cubriendo caminos felices, casos límite, escenarios negativos, seguridad, rendimiento y usabilidad.

### Nombre del sistema
**QA Test Case Generator** — Generador local de casos de prueba impulsado por IA.

### Tipo de tarea
Generación de texto estructurado (JSON) a partir de requisitos en lenguaje natural.

### Usuarios objetivo
- Ingenieros QA / testers de software
- Desarrolladores que necesitan documentar pruebas
- Estudiantes de ingeniería en asignaturas de pruebas de software

### Relevancia para Deep Learning
El proyecto integra el ciclo completo de una aplicación LLM: selección de modelo, orquestación de prompts, evaluación automática con DeepEval y observabilidad con Opik. Demuestra cómo los LLMs locales pueden automatizar tareas cognitivas complejas como el diseño de pruebas de software, sin depender de APIs en la nube.

## 2. Instalación y Configuración Técnica

### 2.1 Verificación de Ollama

In [ ]:
# Verificar que Ollama está instalado y operativo
!ollama --version

In [ ]:
# Listar modelos descargados
!ollama list

> **Análisis:** *(Completar después de ejecutar)* Describir los modelos instalados: nombre, tamaño, cuantización, y el hardware disponible (CPU/GPU/RAM).

### 2.2 Verificación de DeepEval

In [ ]:
!deepeval --version

In [ ]:
# Verificar que DeepEval importa correctamente
from deepeval.metrics import GEval, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
print("DeepEval importado correctamente ✓")

### 2.3 Configuración de Opik

In [ ]:
import os
from dotenv import load_dotenv

# Cargar variables de entorno desde .env
load_dotenv(dotenv_path="../.env")

OPIK_API_KEY     = os.getenv("OPIK_API_KEY", "")
OPIK_PROJECT     = os.getenv("OPIK_PROJECT_NAME", "QA-Test-Generator")
OLLAMA_MODEL     = os.getenv("OLLAMA_MODEL", "llama3.2")

print(f"Proyecto Opik : {OPIK_PROJECT}")
print(f"Modelo Ollama : {OLLAMA_MODEL}")
print(f"API Key Opik  : {'configurada ✓' if OPIK_API_KEY else 'NO configurada ✗'}")

In [ ]:
import opik

# Configurar Opik con la API key
opik.configure(api_key=OPIK_API_KEY, project_name=OPIK_PROJECT)
print("Opik configurado correctamente ✓")

## 3. Prototipo de Interacción — 3 Consultas con Trazas Opik

Las tres consultas a continuación llaman directamente al LLM a través de Ollama y registran cada ejecución como una traza en Opik.

In [ ]:
import ollama
import json
import time

SYSTEM_PROMPT = """Eres un ingeniero QA senior experto. Dado un requisito, genera casos de prueba
estructurados en JSON con las categorías: happy_path, negativo, seguridad, rendimiento.
Responde SOLO con JSON válido, en español."""

USER_STORIES = [
    {
        "id": "US-01",
        "story": """Como usuario, quiero iniciar sesión con mi correo y contraseña para
acceder a mi dashboard personal. El sistema debe bloquear la cuenta tras 3
intentos fallidos y enviar un correo de recuperación."""
    },
    {
        "id": "US-02",
        "story": """Como cliente, quiero agregar productos a mi carrito de compras y proceder
al pago con tarjeta de crédito. El sistema debe validar los datos de la tarjeta
y confirmar la transacción por correo."""
    },
    {
        "id": "US-03",
        "story": """Como administrador, quiero cargar un archivo CSV con hasta 10,000 registros
de usuarios para importar masivamente al sistema. El proceso debe completarse en
menos de 30 segundos y reportar registros fallidos."""
    },
]

resultados = {}

for us in USER_STORIES:
    print(f"\n{'='*60}")
    print(f"Procesando {us['id']}...")
    
    # Crear traza en Opik
    opik_client = opik.Opik()
    trace = opik_client.trace(
        name=f"generate_tc_{us['id']}",
        input={"user_story": us["story"], "model": OLLAMA_MODEL},
        project_name=OPIK_PROJECT,
    )
    
    # Llamar al LLM
    t0 = time.monotonic()
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": us["story"]},
        ],
        options={"temperature": 0.25, "num_ctx": 4096},
    )
    elapsed = time.monotonic() - t0
    content = response["message"]["content"]
    
    # Cerrar traza con output y latencia
    trace.end(output={
        "response_preview": content[:300],
        "elapsed_seconds": round(elapsed, 2),
        "model": OLLAMA_MODEL,
    })
    
    resultados[us["id"]] = {"story": us["story"], "content": content, "elapsed": elapsed}
    print(f"  Tiempo: {elapsed:.1f}s")
    print(f"  Primeros 200 chars: {content[:200]}")

print("\n✓ Las 3 trazas han sido enviadas a Opik")

## 4. Evidencias Opik — 3 Pantallazos con Análisis

### Pantallazo 1 — Vista general del proyecto con trazas registradas

> **[INSERTAR CAPTURA DE PANTALLA AQUÍ]**
> *(Tomar captura del dashboard Opik mostrando las 3 trazas: generate_tc_US-01, US-02, US-03)*

**Análisis:** *(Completar después de tomar la captura)*  
El dashboard de Opik muestra ... campos por traza: nombre, timestamp, duración y estado. Las 3 trazas registradas corresponden a las consultas US-01 (login), US-02 (carrito) y US-03 (carga CSV). Se puede observar que Opik captura automáticamente el input enviado al modelo y el output recibido, lo que permite auditar cualquier interacción con el LLM.

---

### Pantallazo 2 — Detalle de una traza específica

> **[INSERTAR CAPTURA DE PANTALLA AQUÍ]**
> *(Abrir la traza de US-01 y mostrar: input, output, modelo, timestamp, duración)*

**Análisis:** *(Completar después de tomar la captura)*  
La traza de US-01 muestra el texto de la historia de usuario como input y el JSON generado como output. El tiempo de respuesta fue de ... segundos, lo cual es [esperado / sorprendente] para el modelo llama3.2 corriendo localmente en CPU. Este nivel de detalle permite identificar si el modelo tardó más de lo normal, lo que podría indicar saturación del hardware o un prompt demasiado largo.

---

### Pantallazo 3 — Vista de estadísticas básicas

> **[INSERTAR CAPTURA DE PANTALLA AQUÍ]**
> *(Mostrar la vista de estadísticas: número de runs, latencia promedio)*

**Análisis:** *(Completar después de tomar la captura)*  
Las estadísticas muestran ... ejecuciones registradas con una latencia promedio de ... segundos. Con estas vistas podríamos detectar, en entregas futuras, regresiones de rendimiento al cambiar el modelo (ej. pasar de llama3.2 a mistral) o degradación por prompts más largos.

## 5. Evaluación con DeepEval — Test AnswerRelevancyMetric

In [ ]:
import sys
sys.path.insert(0, "..")

from evaluator.metrics import OllamaEvalModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

eval_model = OllamaEvalModel(OLLAMA_MODEL)

relevancy_metric = GEval(
    name="Relevancia de Test Cases",
    criteria="""Evalúa si el output generado (test cases) es directamente relevante
    a la historia de usuario proporcionada como input.
    Penaliza si los casos describen funcionalidades no mencionadas en el requisito.
    Puntúa de 0 a 1, donde 1 significa máxima relevancia.""",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
    threshold=0.7,
)

# Evaluar la primera consulta
us01 = resultados["US-01"]
test_case_01 = LLMTestCase(
    input=us01["story"],
    actual_output=us01["content"][:1000],  # limitar para evaluación
)

print("Ejecutando métrica de relevancia en US-01...")
relevancy_metric.measure(test_case_01)

print(f"\nResultado DeepEval — Relevancia:")
print(f"  Score   : {relevancy_metric.score:.3f}")
print(f"  Passed  : {relevancy_metric.is_successful()}")
print(f"  Reason  : {getattr(relevancy_metric, 'reason', 'N/A')}")

### Análisis del resultado DeepEval

> **[INSERTAR CAPTURA DE PANTALLA DE LA CELDA CON RESULTADO AQUÍ]**

**Análisis (3-5 líneas):** *(Completar después de ejecutar)*  
El score de relevancia obtenido fue de ..., lo que indica que el modelo [generó / no generó] casos bien alineados con el requisito de login. El resultado [superó / no superó] el umbral de 0.7. La razón proporcionada por DeepEval fue: "...". Este resultado sugiere que [agregar conclusión sobre la calidad del output del LLM].

## 6. Diagrama de Arquitectura del Sistema Completo

```
┌─────────────────────────────────────────────────────────────┐
│                    USUARIO                                  │
└─────────────────────┬───────────────────────────────────────┘
                      │
┌─────────────────────▼───────────────────────────────────────┐
│           FRONTEND (HTML/CSS/JS Vanilla)                    │
│  - Formulario de historia de usuario                        │
│  - Selector de modelo y temperatura                         │
│  - Visualización de casos (tabs: TC / Edge / Bugs / Coverage)│
│  - Exportación: JSON, CSV, Markdown, XLSX                   │
└─────────────────────┬───────────────────────────────────────┘
                      │ HTTP (SSE/REST)
┌─────────────────────▼───────────────────────────────────────┐
│           BACKEND (FastAPI + Python)                        │
│  ┌──────────────────────────────────────────────────────┐   │
│  │ routes/generate.py → POST /generate/stream           │   │
│  │ routes/evaluate.py → POST /evaluate                  │   │
│  ├──────────────────────────────────────────────────────┤   │
│  │ services/llm_service.py  ← @opik.track               │   │
│  │   · Prompt del sistema (9 instrucciones)             │   │
│  │   · Llamada a Ollama                                 │   │
│  │   · Parseo y reparación de JSON                      │   │
│  │ services/rag_service.py  [CORTE 2]                   │   │
│  │   · Embeddings: nomic-embed-text (Ollama)            │   │
│  │   · Vector Store: ChromaDB                           │   │
│  │   · Búsqueda semántica sobre corpus QA               │   │
│  │ services/agent_service.py [CORTE 2]                  │   │
│  │   · Agente 1: Generador (CrewAI)                     │   │
│  │   · Agente 2: Revisor de Calidad (CrewAI)            │   │
│  │ services/eval_service.py                             │   │
│  │   · DeepEval: GEval x3 métricas                      │   │
│  └──────────────────────────────────────────────────────┘   │
└───────┬──────────────────┬──────────────────┬───────────────┘
        │                  │                  │
┌───────▼───┐    ┌─────────▼──────┐  ┌───────▼────────────┐
│  OLLAMA   │    │    OPIK        │  │   DEEPEVAL         │
│  (Local)  │    │  (Dashboard)   │  │  (Evaluación)      │
│ llama3.2  │    │  Trazas/Runs   │  │  Coverage/Relevancy│
│ mistral   │    │  Experimentos  │  │  Consistency       │
│ phi3      │    │  Latencia      │  │  [+2 métricas C2]  │
└───────────┘    └────────────────┘  └────────────────────┘
```

**Nota:** Los módulos marcados con `[CORTE 2]` serán implementados en la segunda entrega. La arquitectura actual (Corte 1) cubre todo el flujo excepto RAG y agentes.

## 7. Plan de Trabajo — Corte 2

| Semana | Sprint | Entregable |
|--------|--------|------------|
| Sem. 2 (abr 26 - may 2) | Sprint Acad. 1: RAG | `rag_service.py` + ChromaDB + `scripts/build_index.py` |
| Sem. 3 (may 3-9) | Sprint Acad. 2: Agentes | `agent_service.py` (CrewAI Generador + Revisor) + endpoint `/generate/agents` |
| Sem. 4 (may 10-16) | Sprint Acad. 3: Experimento | 8-15 user stories, Opik experimentos, DeepEval 3+ métricas |
| Sem. 5 (may 17-23) | Entrega Corte 2 | Notebook `corte2_evaluacion.ipynb`, video 5 min, manual de uso |

### Modelos a comparar en Corte 2
- **Configuración A:** `llama3.2` sin RAG (baseline actual)
- **Configuración B:** `mistral` con RAG sobre corpus de buenas prácticas QA

### Métricas DeepEval para Corte 2
1. `Test Coverage` (GEval) — ya implementada
2. `Test Relevancy` (GEval) — ya implementada  
3. `Test Consistency` (GEval) — ya implementada
4. `AnswerRelevancyMetric` — por implementar en Corte 2
5. `FaithfulnessMetric` — por implementar en Corte 2 (mide si la respuesta es fiel al contexto RAG)

### Experimento formal planeado
- 10 historias de usuario de distintos dominios (login, pagos, carrito, API, carga de archivos, búsqueda, notificaciones, perfil, reportes, onboarding)
- Cada historia se procesa con Configuración A y B
- Total: 20 ejecuciones registradas en Opik
- Análisis crítico: calidad vs. velocidad, detección de alucinaciones, propuesta de mejora

## 8. Evidencias de Instalación

### Hardware utilizado

In [ ]:
import platform
import psutil

print(f"SO         : {platform.system()} {platform.release()}")
print(f"CPU        : {platform.processor()}")
print(f"RAM total  : {psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"RAM libre  : {psutil.virtual_memory().available / (1024**3):.1f} GB")
print(f"Python     : {platform.python_version()}")

In [ ]:
# Versiones de dependencias clave
import fastapi, ollama, pydantic, opik
print(f"FastAPI  : {fastapi.__version__}")
print(f"Ollama   : {ollama.__version__}")
print(f"Pydantic : {pydantic.__version__}")
print(f"Opik     : {opik.__version__}")

---

## Repositorio

**URL:** https://github.com/JesusAndres2512/Proyecto_DL_QA  
**Rama principal:** `main`  
**Rama de desarrollo actual:** `jesusasus`

---
*Generado para la entrega del Corte 1 — Materia Deep Learning*